# BindingDB + TTD Demo

This notebook follows the Cheng-style modality layout:
1. Setup
2. Direct SQL
3. REST API
4. MCP tools
5. Agentic Q&A

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import requests

NODE_URL = os.getenv("TTD_NODE_URL", "http://agentic-dataset-bindingdb-ttd:8006")
DB_PATH = Path(os.getenv("TTD_DB_PATH", "/app/data/ttd.sqlite3"))

print(f"NODE_URL={NODE_URL}")
print(f"DB_PATH={DB_PATH}")

## 1) Setup

Check service health and basic dataset summary.

In [ ]:
summary = requests.get(f"{NODE_URL}/data/summary", timeout=20).json()
summary

## 2) Direct SQL

Read directly from local SQLite for quick schema/data checks.

In [ ]:
if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT c.name, t.target_name, i.metric_type, i.metric_value, i.metric_unit FROM interactions i JOIN compounds c ON c.id=i.compound_id JOIN targets t ON t.id=i.target_id LIMIT 10")
    rows = cur.fetchall()
    conn.close()
    rows
else:
    print("DB file not found. Use REST sections below (node may hold DB in container path).")

## 3) REST API

Query through node endpoints (`/data/query`, `/ttd/search`, `/ttd/target/{uniprot_id}`).

In [ ]:
query_payload = {"sql": "SELECT name, source FROM compounds"}
query_resp = requests.post(f"{NODE_URL}/data/query", json=query_payload, timeout=20)
search_resp = requests.get(f"{NODE_URL}/ttd/search", params={"q": "EGFR"}, timeout=20)
target_resp = requests.get(f"{NODE_URL}/ttd/target/P00533", timeout=20)
{
    "query_status": query_resp.status_code,
    "query_sample": query_resp.json().get("rows", [])[:3],
    "search_count": search_resp.json().get("count"),
    "target_count": target_resp.json().get("count"),
}

## 4) MCP Tools

Call MCP tool routes for table discovery and domain search.

In [ ]:
tools = requests.get(f"{NODE_URL}/mcp/tools", timeout=20).json()
tool_names = [t["name"] for t in tools.get("tools", [])]
tool_names

In [ ]:
call_payload = {"tool": "search_compounds", "args": {"query": "EGFR", "limit": 5}}
mcp_search = requests.post(f"{NODE_URL}/mcp/call", json=call_payload, timeout=20)
mcp_search.status_code, mcp_search.json()

## 5) Agentic Q&A

Use the A2A endpoint to ask natural-language questions grounded in this node.

In [ ]:
rpc_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "message/send",
    "params": {
        "message": {
            "parts": [{"kind": "text", "text": "Which compounds in this dataset target EGFR and what are their affinity metrics?"}]
        }
    }
}
rpc_resp = requests.post(f"{NODE_URL}/rpc", json=rpc_payload, timeout=60)
rpc_resp.status_code, rpc_resp.json()